# SmartHire — Skill Gap / Fit Extension
The official brief marks the fit predictor as optional. The core project uses skill-gap analysis instead.

In [28]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [29]:
jobs_df = pd.read_csv("../data/processed/jobs_clean.csv")

print("Dataset Shape:", jobs_df.shape)

Dataset Shape: (21851, 11)


In [30]:
sample_resume = """
Computer Science Engineering student with strong knowledge of
Python, Java, C++, SQL, Machine Learning, Artificial Intelligence,
Natural Language Processing and Data Science.

Skills include Pandas, NumPy, Scikit-learn, TensorFlow,
Matplotlib, Streamlit and Git.

Projects include Resume Classification, House Price Prediction,
Heart Disease Prediction and an NLP Chatbot.

Interested in Machine Learning Engineer, AI Engineer,
Software Engineer and Data Scientist roles.
"""

In [31]:
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

job_vectors = tfidf.fit_transform(
    jobs_df["job_text"].fillna("")
)

resume_vector = tfidf.transform([sample_resume])

similarity_scores = cosine_similarity(
    resume_vector,
    job_vectors
).flatten()

jobs_df["text_similarity"] = similarity_scores

print("Similarity calculated successfully!")

Similarity calculated successfully!


In [32]:
skills = [
    "python",
    "java",
    "c++",
    "c",
    "sql",
    "machine learning",
    "artificial intelligence",
    "data science",
    "deep learning",
    "natural language processing",
    "tensorflow",
    "pytorch",
    "pandas",
    "numpy",
    "scikit-learn",
    "streamlit",
    "git",
    "javascript",
    "html",
    "css"
]

In [33]:
resume_lower = sample_resume.lower()

resume_skills = {
    skill for skill in skills
    if skill in resume_lower
}

print("Resume Skills:")
print(resume_skills)

Resume Skills:
{'numpy', 'sql', 'streamlit', 'machine learning', 'natural language processing', 'tensorflow', 'c++', 'git', 'pandas', 'c', 'scikit-learn', 'java', 'artificial intelligence', 'data science', 'python'}


In [34]:
def calculate_skill_overlap(job_text):

    job_text = job_text.lower()

    job_skills = {
        skill for skill in skills
        if skill in job_text
    }

    if len(job_skills) == 0:
        return 0

    matching_skills = resume_skills.intersection(job_skills)

    return len(matching_skills) / len(job_skills)

In [35]:
jobs_df["skill_overlap"] = jobs_df["job_text"].apply(
    calculate_skill_overlap
)

print("Skill overlap calculated successfully!")

Skill overlap calculated successfully!


In [36]:
jobs_df["fit_score"] = (
    0.6 * jobs_df["text_similarity"] +
    0.4 * jobs_df["skill_overlap"]
)

jobs_df["fit_score"] = jobs_df["fit_score"] * 100

In [37]:
top_fit_jobs = jobs_df.sort_values(
    "fit_score",
    ascending=False
).head(10)

print(
    top_fit_jobs[
        [
            "jobtitle",
            "company",
            "joblocation_address",
            "fit_score"
        ]
    ]
)

                                                jobtitle  \
10603                    data scientist machine learning   
17338               software engineer - machine learning   
5799                     data scientist-machine learning   
21603                                     data scientist   
9388              engineering manager - machine learning   
11442               data scientist senior data scientist   
19986                       team lead - machine learning   
15524              data scientist - machine learning/nlp   
10830  sr scientist/ nlp/ machine learning/ data mini...   
21790  machine learning / nlp /big data expert for r&...   

                                                 company  joblocation_address  \
10603                      brillio technologies pvt. ltd  bengaluru/bangalore   
17338                                              quora               mumbai   
5799                        rinalytics advisors pvt. ltd  bengaluru/bangalore   
21603  careerne

In [38]:
best_job = top_fit_jobs.iloc[0]

print("Job Title:", best_job["jobtitle"])
print("Company:", best_job["company"])
print("Fit Score:", round(best_job["fit_score"], 2), "%")

Job Title: data scientist machine learning
Company: brillio technologies pvt. ltd
Fit Score: 68.5 %


In [39]:
score = best_job["fit_score"]

if score >= 70:
    fit_level = "Excellent Fit"
elif score >= 50:
    fit_level = "Good Fit"
elif score >= 30:
    fit_level = "Moderate Fit"
else:
    fit_level = "Low Fit"

print("Fit Level:", fit_level)

Fit Level: Good Fit


In [40]:
top_fit_jobs.to_csv(
    "../reports/fit_prediction_results.csv",
    index=False
)

print("Fit prediction results saved successfully!")

Fit prediction results saved successfully!
